# 📊 Bulk Evaluation — Full Pipeline

Evaluate the entire detection pipeline on a **folder of images** in one go.

### Ground-truth convention

The expected label is inferred from the **filename prefix**:

| Prefix | Expected label |
|--------|----------------|
| `fake` or `ai` | **AI Generated** |
| `real` or `human` | **Real** |

### What gets evaluated

| Column | Source |
|--------|--------|
| **Base model** | `dima806/ai_vs_human_generated_image_detection` (no delta) |
| **Fine-tuned model** | Base + weight delta |
| **Metadata module** | EXIF / C2PA / binary marker scoring |
| **Fused (pipeline)** | Fusion of metadata + fine-tuned visual via the selected strategy |
| **Deepfake module** | Only run on fused-positive images |

## 1. Setup & Imports

In [1]:
import sys
import os
import glob
import time
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

# Add project root to sys.path
PROJECT_ROOT = str(Path.cwd().parent.parent)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from PIL import Image
from IPython.display import display, HTML

# Our modules
from src.metadata_module import analyse_image
from src.visual_module.visual_classifier import VisualClassifier
from src.deepfake_module.deepfake_classifier import DeepfakeClassifier
from src.integration_pipeline.fusion import (
    get_fusion_strategy,
    extract_visual_ai_probability,
)

print("✅ All imports successful.")

W0525 14:39:29.877000 15204 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


✅ All imports successful.


## 2. Configuration

Set the folder to evaluate and the fusion strategy parameters here.

In [2]:
# ── Folder to evaluate ──────────────────────────────────────────────────
IMAGE_FOLDER = os.path.join(PROJECT_ROOT, "data", "sample_images")

# ── Supported extensions ────────────────────────────────────────────────
EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".tiff"}

# ── Fusion strategy ─────────────────────────────────────────────────────
# Options: "weighted_average", "conservative_threshold", "bayesian"
FUSION_STRATEGY = "weighted_average"
FUSION_KWARGS = dict(
    w_meta=0.3,
    w_visual=0.7,
    decision_threshold=0.55,
    visual_accuracy=0.92,
)

# ── Fine-tuned model delta path ─────────────────────────────────────────
VISUAL_DELTA_PATH = os.path.join(
    PROJECT_ROOT, "src", "visual_module", "fine_tuned_model_delta", "weight_delta.pt"
)

# ── Deepfake module paths ───────────────────────────────────────────────
DEEPFAKE_INDEX_PATH = os.path.join(PROJECT_ROOT, "src", "deepfake_module", "models", "landmarks_index.faiss")
DEEPFAKE_META_PATH  = os.path.join(PROJECT_ROOT, "src", "deepfake_module", "models", "landmarks_metadata.json")

# ── Run deepfake module on AI-positive images? ──────────────────────────
RUN_DEEPFAKE = True

print(f"📂 Image folder : {IMAGE_FOLDER}")
print(f"🧮 Fusion        : {FUSION_STRATEGY}")
print(f"🎭 Deepfake      : {'Enabled' if RUN_DEEPFAKE else 'Disabled'}")

📂 Image folder : /Users/davidscerri/Library/Mobile Documents/com~apple~CloudDocs/Studies/Masters/Proof-of-Concept/Seeing-Through-Deepfakes/data/sample_images
🧮 Fusion        : weighted_average
🎭 Deepfake      : Enabled


## 3. Load Models

We load **two** visual classifiers — base and fine-tuned — plus the deepfake classifier.

In [3]:
# ── Base visual classifier (no delta) ───────────────────────────────────
print("Loading Base Visual Classifier...")
base_classifier = VisualClassifier(
    model_name_or_path="dima806/ai_vs_human_generated_image_detection",
    delta_path=None,
)
print("✅ Base Visual Classifier ready.\n")

# ── Fine-tuned visual classifier (base + delta) ─────────────────────────
print("Loading Fine-tuned Visual Classifier...")
finetuned_classifier = VisualClassifier(
    model_name_or_path="dima806/ai_vs_human_generated_image_detection",
    delta_path=VISUAL_DELTA_PATH,
)
print("✅ Fine-tuned Visual Classifier ready.\n")

# ── Deepfake classifier ─────────────────────────────────────────────────
deepfake_classifier = None
if RUN_DEEPFAKE:
    print("Loading Deepfake Classifier...")
    deepfake_classifier = DeepfakeClassifier(
        index_path=DEEPFAKE_INDEX_PATH,
        metadata_path=DEEPFAKE_META_PATH,
    )
    print("✅ Deepfake Classifier ready.")

Loading Base Visual Classifier...


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

✅ Base Visual Classifier ready.

Loading Fine-tuned Visual Classifier...


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Applying weight delta from '/Users/davidscerri/Library/Mobile Documents/com~apple~CloudDocs/Studies/Masters/Proof-of-Concept/Seeing-Through-Deepfakes/src/visual_module/fine_tuned_model_delta/weight_delta.pt'...
Delta applied successfully.
✅ Fine-tuned Visual Classifier ready.

Loading Deepfake Classifier...
Loading Face Detection model: MTCNN
Loading Scene model: birder-project/rope_vit_reg4_b14_capi-places365


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading Landmark Retrieval model: facebook/dinov2-base


Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

Loading FAISS index from /Users/davidscerri/Library/Mobile Documents/com~apple~CloudDocs/Studies/Masters/Proof-of-Concept/Seeing-Through-Deepfakes/src/deepfake_module/models/landmarks_index.faiss...
Landmark index loaded successfully.
✅ Deepfake Classifier ready.


## 4. Run Bulk Evaluation

Iterates over every image in the folder, runs all modules, and collects results.

In [4]:
def infer_ground_truth(filename: str) -> str:
    """
    Derive the expected label from the filename.
    Returns 'AI Generated' or 'Real' (or 'Unknown' if no prefix matches).
    """
    name = filename.lower()
    if name.startswith("fake") or name.startswith("ai"):
        return "AI Generated"
    elif name.startswith("real") or name.startswith("human"):
        return "Real"
    return "Unknown"


def evaluate_folder(folder_path: str) -> list[dict]:
    """
    Run the full evaluation pipeline on every image in `folder_path`.
    Returns a list of result dictionaries.
    """
    # Collect image files
    image_files = sorted([
        f for f in os.listdir(folder_path)
        if Path(f).suffix.lower() in EXTENSIONS and not f.startswith(".")
    ])

    if not image_files:
        print(f"⚠️ No image files found in {folder_path}")
        return []

    print(f"Found {len(image_files)} images to evaluate.\n")

    strategy = get_fusion_strategy(FUSION_STRATEGY, **FUSION_KWARGS)
    results = []

    for i, filename in enumerate(image_files, 1):
        filepath = os.path.join(folder_path, filename)
        expected = infer_ground_truth(filename)

        print(f"[{i}/{len(image_files)}] {filename}  (expected: {expected})")

        # Load image
        pil_image = Image.open(filepath)
        if pil_image.mode != "RGB":
            pil_image = pil_image.convert("RGB")

        # ── Base visual classifier ────────────────────────────────────
        base_result = base_classifier.predict(pil_image)
        base_ai_prob = extract_visual_ai_probability(base_result)

        # ── Fine-tuned visual classifier ──────────────────────────────
        ft_result = finetuned_classifier.predict(pil_image)
        ft_ai_prob = extract_visual_ai_probability(ft_result)

        # ── Metadata module ───────────────────────────────────────────
        meta_result = analyse_image(filepath)

        # ── Fusion (metadata + fine-tuned visual) ─────────────────────
        fusion_result = strategy.fuse(
            metadata_ai_prob=meta_result.ai_probability,
            visual_ai_prob=ft_ai_prob,
        )

        # ── Deepfake module (conditional) ─────────────────────────────
        df_decision = "—"
        has_face = False
        has_place = False
        if RUN_DEEPFAKE and deepfake_classifier and fusion_result.is_ai:
            df_result = deepfake_classifier.predict(pil_image)
            da = df_result.get("deepfake_analysis")
            if da:
                has_face = da.get("has_face", False)
                has_place = da.get("has_place", False)
            df_decision = df_result.get("final_decision", "N/A")

        # ── Build final verdict ───────────────────────────────────────
        if fusion_result.is_ai:
            if has_face or has_place:
                verdict = "Probable Deepfake"
            else:
                verdict = "AI Generated"
            fused_label = "AI Generated"
        else:
            verdict = "Real"
            fused_label = "Real"

        results.append({
            "filename": filename,
            "expected": expected,
            # Base model
            "base_prediction": base_result["prediction"],
            "base_confidence": base_result["confidence"],
            "base_ai_prob": base_ai_prob,
            "base_correct": base_result["prediction"] == expected,
            # Fine-tuned model
            "ft_prediction": ft_result["prediction"],
            "ft_confidence": ft_result["confidence"],
            "ft_ai_prob": ft_ai_prob,
            "ft_correct": ft_result["prediction"] == expected,
            # Metadata module
            "meta_ai_prob": meta_result.ai_probability,
            "meta_decision": meta_result.decision,
            # Fusion
            "fused_ai_prob": fusion_result.ai_probability,
            "fused_label": fused_label,
            "fused_correct": fused_label == expected,
            # Deepfake
            "has_face": has_face,
            "has_place": has_place,
            "verdict": verdict,
            "verdict_correct": (verdict != "Real") == (expected == "AI Generated"),
        })

        # Quick inline summary
        base_icon = "✅" if results[-1]["base_correct"] else "❌"
        ft_icon   = "✅" if results[-1]["ft_correct"]   else "❌"
        fused_icon = "✅" if results[-1]["fused_correct"] else "❌"
        print(f"   Base: {base_result['prediction']:14s} ({base_ai_prob:.2%}) {base_icon}")
        print(f"   F.T.: {ft_result['prediction']:14s} ({ft_ai_prob:.2%}) {ft_icon}")
        print(f"   Fused: {fused_label:13s} ({fusion_result.ai_probability:.2%}) {fused_icon}")
        if df_decision != "—":
            print(f"   Deepfake: {df_decision}")
        print()

    return results


# ── Run it ───────────────────────────────────────────────────────────────
start_time = time.time()
results = evaluate_folder(IMAGE_FOLDER)
elapsed = time.time() - start_time
print(f"\n⏱️ Evaluation completed in {elapsed:.1f}s")

Found 8 images to evaluate.

[1/8] fake+metadata+place-face.png  (expected: AI Generated)
   Base: Real           (0.94%) ❌
   F.T.: AI Generated   (53.24%) ✅
   Fused: AI Generated  (67.78%) ✅
   Deepfake: Potential Deepfake (AI Image containing Face/Place)

[2/8] fake+metadata-place+face.png  (expected: AI Generated)
   Base: Real           (0.80%) ❌
   F.T.: AI Generated   (96.98%) ✅
   Fused: AI Generated  (97.62%) ✅
   Deepfake: Potential Deepfake (AI Image containing Face/Place)

[3/8] fake+metadata-place-face.png  (expected: AI Generated)
   Base: Real           (0.99%) ❌
   F.T.: AI Generated   (54.00%) ✅
   Fused: AI Generated  (68.30%) ✅
   Deepfake: Potential Deepfake (AI Image containing Face/Place)

[4/8] fake-metadata+place-face.png  (expected: AI Generated)
   Base: Real           (0.94%) ❌
   F.T.: AI Generated   (53.24%) ✅
   Fused: Real          (53.16%) ❌

[5/8] fake-metadata-place+face.png  (expected: AI Generated)
   Base: Real           (0.80%) ❌
   F.T.: AI Gener

## 5. Results Table

A formatted comparison table showing every image's expected vs predicted labels.

In [5]:
def render_results_table(results: list[dict]):
    """Render a rich HTML results table."""
    if not results:
        print("No results to display.")
        return

    html = """
    <style>
        .eval-table {
            border-collapse: collapse;
            width: 100%;
            font-size: 13px;
            font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
        }
        .eval-table th {
            background: #2c3e50;
            color: white;
            padding: 10px 12px;
            text-align: left;
            font-weight: 600;
            position: sticky;
            top: 0;
        }
        .eval-table td {
            padding: 8px 12px;
            border-bottom: 1px solid #e0e0e0;
        }
        .eval-table tr:nth-child(even) { background: #f8f9fa; }
        .eval-table tr:hover { background: #e8f4fd; }
        .correct { color: #27ae60; font-weight: bold; }
        .incorrect { color: #e74c3c; font-weight: bold; }
        .prob-high { color: #e74c3c; }
        .prob-low  { color: #27ae60; }
        .label-ai  { background: #fdecea; padding: 2px 6px; border-radius: 4px; }
        .label-real { background: #eafaf1; padding: 2px 6px; border-radius: 4px; }
    </style>
    <h3>📋 Per-Image Results</h3>
    <div style="overflow-x: auto;">
    <table class="eval-table">
    <thead>
        <tr>
            <th>#</th>
            <th>Filename</th>
            <th>Expected</th>
            <th>Base Model</th>
            <th>Base P(AI)</th>
            <th></th>
            <th>Fine-tuned</th>
            <th>F.T. P(AI)</th>
            <th></th>
            <th>Meta P(AI)</th>
            <th>Fused</th>
            <th>Fused P(AI)</th>
            <th></th>
            <th>Verdict</th>
        </tr>
    </thead>
    <tbody>
    """

    for i, r in enumerate(results, 1):
        exp_cls = "label-ai" if r["expected"] == "AI Generated" else "label-real"

        base_cls = "correct" if r["base_correct"] else "incorrect"
        base_icon = "✅" if r["base_correct"] else "❌"

        ft_cls = "correct" if r["ft_correct"] else "incorrect"
        ft_icon = "✅" if r["ft_correct"] else "❌"

        fused_cls = "correct" if r["fused_correct"] else "incorrect"
        fused_icon = "✅" if r["fused_correct"] else "❌"

        prob_cls = lambda p: "prob-high" if p > 0.5 else "prob-low"

        html += f"""
        <tr>
            <td>{i}</td>
            <td><code>{r['filename']}</code></td>
            <td><span class="{exp_cls}">{r['expected']}</span></td>
            <td class="{base_cls}">{r['base_prediction']}</td>
            <td class="{prob_cls(r['base_ai_prob'])}">{r['base_ai_prob']:.2%}</td>
            <td>{base_icon}</td>
            <td class="{ft_cls}">{r['ft_prediction']}</td>
            <td class="{prob_cls(r['ft_ai_prob'])}">{r['ft_ai_prob']:.2%}</td>
            <td>{ft_icon}</td>
            <td class="{prob_cls(r['meta_ai_prob'])}">{r['meta_ai_prob']:.2%}</td>
            <td class="{fused_cls}">{r['fused_label']}</td>
            <td class="{prob_cls(r['fused_ai_prob'])}">{r['fused_ai_prob']:.2%}</td>
            <td>{fused_icon}</td>
            <td>{r['verdict']}</td>
        </tr>
        """

    html += "</tbody></table></div>"
    display(HTML(html))


render_results_table(results)

#,Filename,Expected,Base Model,Base P(AI),,Fine-tuned,F.T. P(AI),,Meta P(AI),Fused,Fused P(AI),,Verdict
1,fake+metadata+place-face.png,AI Generated,Real,0.94%,❌,AI Generated,53.24%,✅,99.00%,AI Generated,67.78%,✅,Probable Deepfake
2,fake+metadata-place+face.png,AI Generated,Real,0.80%,❌,AI Generated,96.98%,✅,99.00%,AI Generated,97.62%,✅,Probable Deepfake
3,fake+metadata-place-face.png,AI Generated,Real,0.99%,❌,AI Generated,54.00%,✅,99.00%,AI Generated,68.30%,✅,Probable Deepfake
4,fake-metadata+place-face.png,AI Generated,Real,0.94%,❌,AI Generated,53.24%,✅,53.00%,Real,53.16%,❌,Real
5,fake-metadata-place+face.png,AI Generated,Real,0.80%,❌,AI Generated,96.98%,✅,50.00%,AI Generated,82.05%,✅,Probable Deepfake
6,fake-metadata-place-face.png,AI Generated,Real,0.99%,❌,AI Generated,54.00%,✅,50.00%,Real,52.73%,❌,Real
7,real+metadata.jpeg,Real,Real,0.97%,✅,AI Generated,91.05%,❌,35.00%,AI Generated,73.24%,❌,Probable Deepfake
8,real-metadata.jpeg,Real,Real,0.97%,✅,AI Generated,91.05%,❌,50.00%,AI Generated,78.00%,❌,Probable Deepfake


## 6. Aggregate Accuracy Metrics

Overall accuracy, plus breakdowns by class (AI Generated vs Real).

In [6]:
def compute_accuracy_metrics(results: list[dict]):
    """Compute and display aggregate accuracy metrics."""
    if not results:
        print("No results.")
        return

    # Filter out unknowns
    labelled = [r for r in results if r["expected"] != "Unknown"]
    if not labelled:
        print("⚠️ No labelled images found (filenames must start with 'fake'/'ai' or 'real'/'human').")
        return

    total = len(labelled)
    ai_images = [r for r in labelled if r["expected"] == "AI Generated"]
    real_images = [r for r in labelled if r["expected"] == "Real"]

    def acc(items, key):
        if not items:
            return 0.0
        return sum(1 for r in items if r[key]) / len(items)

    # ── Summary table ─────────────────────────────────────────────────
    html = """
    <h3>📊 Aggregate Accuracy</h3>
    <table style="border-collapse:collapse; font-size:14px; font-family: -apple-system, sans-serif;">
    <thead>
        <tr style="background:#34495e; color:white;">
            <th style="padding:10px 16px; text-align:left;">Model / Module</th>
            <th style="padding:10px 16px; text-align:right;">Overall</th>
            <th style="padding:10px 16px; text-align:right;">AI Images</th>
            <th style="padding:10px 16px; text-align:right;">Real Images</th>
        </tr>
    </thead>
    <tbody>
    """

    rows = [
        ("Base Visual Classifier",   "base_correct"),
        ("Fine-tuned Visual Classifier", "ft_correct"),
        ("Fused Pipeline",           "fused_correct"),
        ("Full Verdict",             "verdict_correct"),
    ]

    for label, key in rows:
        overall = acc(labelled, key)
        ai_acc  = acc(ai_images, key)
        real_acc = acc(real_images, key)

        def colour(v):
            if v >= 0.8: return "#27ae60"
            if v >= 0.5: return "#f39c12"
            return "#e74c3c"

        html += f"""
        <tr>
            <td style="padding:8px 16px; border-bottom:1px solid #eee;">{label}</td>
            <td style="padding:8px 16px; text-align:right; border-bottom:1px solid #eee;
                       color:{colour(overall)}; font-weight:bold;">{overall:.1%}</td>
            <td style="padding:8px 16px; text-align:right; border-bottom:1px solid #eee;
                       color:{colour(ai_acc)};">{ai_acc:.1%} ({sum(1 for r in ai_images if r[key])}/{len(ai_images)})</td>
            <td style="padding:8px 16px; text-align:right; border-bottom:1px solid #eee;
                       color:{colour(real_acc)};">{real_acc:.1%} ({sum(1 for r in real_images if r[key])}/{len(real_images)})</td>
        </tr>
        """

    html += "</tbody></table>"

    # ── Counts ────────────────────────────────────────────────────────
    html += f"""
    <p style="font-size:13px; color:#666; margin-top:12px;">
        Total images evaluated: <b>{total}</b>
        &nbsp;|&nbsp; AI: <b>{len(ai_images)}</b>
        &nbsp;|&nbsp; Real: <b>{len(real_images)}</b>
    </p>
    """

    display(HTML(html))


compute_accuracy_metrics(results)

Model / Module,Overall,AI Images,Real Images
Base Visual Classifier,25.0%,0.0% (0/6),100.0% (2/2)
Fine-tuned Visual Classifier,75.0%,100.0% (6/6),0.0% (0/2)
Fused Pipeline,50.0%,66.7% (4/6),0.0% (0/2)
Full Verdict,50.0%,66.7% (4/6),0.0% (0/2)


## 7. Confusion Matrix (Visual)

A quick visual confusion matrix for each model stage.

In [7]:
def render_confusion_matrices(results: list[dict]):
    """Render side-by-side confusion matrices for base, fine-tuned, and fused."""
    labelled = [r for r in results if r["expected"] != "Unknown"]
    if not labelled:
        return

    def confusion(items, pred_key, positive="AI Generated"):
        tp = sum(1 for r in items if r["expected"] == positive and r[pred_key] == positive)
        fp = sum(1 for r in items if r["expected"] != positive and r[pred_key] == positive)
        fn = sum(1 for r in items if r["expected"] == positive and r[pred_key] != positive)
        tn = sum(1 for r in items if r["expected"] != positive and r[pred_key] != positive)
        return tp, fp, fn, tn

    def matrix_html(title, tp, fp, fn, tn):
        total = tp + fp + fn + tn
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

        return f"""
        <div style="display:inline-block; margin:10px 20px 10px 0; vertical-align:top;">
            <h4 style="margin-bottom:8px;">{title}</h4>
            <table style="border-collapse:collapse; font-size:13px; text-align:center;">
                <tr>
                    <td style="padding:6px;"></td>
                    <td style="padding:6px;"></td>
                    <td colspan="2" style="padding:6px; font-weight:bold; background:#ecf0f1;">Predicted</td>
                </tr>
                <tr>
                    <td style="padding:6px;"></td>
                    <td style="padding:6px;"></td>
                    <td style="padding:6px 12px; font-weight:bold; background:#ecf0f1;">AI</td>
                    <td style="padding:6px 12px; font-weight:bold; background:#ecf0f1;">Real</td>
                </tr>
                <tr>
                    <td rowspan="2" style="padding:6px; font-weight:bold; background:#ecf0f1; writing-mode:vertical-lr;">Actual</td>
                    <td style="padding:6px; font-weight:bold; background:#ecf0f1;">AI</td>
                    <td style="padding:12px; background:#d5f5e3; font-weight:bold; font-size:16px;">{tp}</td>
                    <td style="padding:12px; background:#fadbd8; font-size:16px;">{fn}</td>
                </tr>
                <tr>
                    <td style="padding:6px; font-weight:bold; background:#ecf0f1;">Real</td>
                    <td style="padding:12px; background:#fadbd8; font-size:16px;">{fp}</td>
                    <td style="padding:12px; background:#d5f5e3; font-weight:bold; font-size:16px;">{tn}</td>
                </tr>
            </table>
            <p style="font-size:12px; color:#666; margin-top:6px;">
                Precision: {precision:.2%} &nbsp;|&nbsp; Recall: {recall:.2%} &nbsp;|&nbsp; F1: {f1:.2%}
            </p>
        </div>
        """

    # Compute confusion matrices
    tp_b, fp_b, fn_b, tn_b = confusion(labelled, "base_prediction")
    tp_f, fp_f, fn_f, tn_f = confusion(labelled, "ft_prediction")
    tp_fu, fp_fu, fn_fu, tn_fu = confusion(labelled, "fused_label")

    html = "<h3>🔢 Confusion Matrices</h3><div>"
    html += matrix_html("Base Model", tp_b, fp_b, fn_b, tn_b)
    html += matrix_html("Fine-tuned Model", tp_f, fp_f, fn_f, tn_f)
    html += matrix_html("Fused Pipeline", tp_fu, fp_fu, fn_fu, tn_fu)
    html += "</div>"

    display(HTML(html))


render_confusion_matrices(results)

## 8. Export Results (Optional)

Save the results to a CSV file for further analysis.

In [8]:
import csv

OUTPUT_CSV = os.path.join(PROJECT_ROOT, "outputs", "bulk_evaluation_results.csv")

if results:
    os.makedirs(os.path.dirname(OUTPUT_CSV), exist_ok=True)
    with open(OUTPUT_CSV, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=results[0].keys())
        writer.writeheader()
        writer.writerows(results)
    print(f"✅ Results exported to: {OUTPUT_CSV}")
else:
    print("No results to export.")

✅ Results exported to: /Users/davidscerri/Library/Mobile Documents/com~apple~CloudDocs/Studies/Masters/Proof-of-Concept/Seeing-Through-Deepfakes/outputs/bulk_evaluation_results.csv
